# Huawei Technology Lab 1 — MindSpore Fundamentals
**Healthcare framing:** structured teaching data → tensor → model → prediction.

This lab uses Huawei MindSpore directly. Synthetic data are used so no patient information is required.

**Educational prototype — not for clinical diagnosis or treatment.**

## 1. Import MindSpore and inspect tensors

In [ ]:
import numpy as np
import mindspore as ms
from mindspore import nn, ops
import mindspore.dataset as ds

ms.set_seed(42)
print('MindSpore version:', ms.__version__)
print('Device target:', ms.get_context('device_target'))

patient_example = ms.Tensor([120.0, 80.0, 37.0], ms.float32)
print('Tensor:', patient_example)
print('Shape:', patient_example.shape)

## 2. Create a safe synthetic teaching dataset
The target is intentionally artificial. The goal is to learn the workflow, not medicine.

In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(size=(256, 10)).astype(np.float32)
y = (X[:,0] + 0.8*X[:,1] - 0.5*X[:,2] > 0).astype(np.int32)

train_ds = ds.NumpySlicesDataset((X[:200], y[:200]), column_names=['features','label'], shuffle=True).batch(32)
test_x = ms.Tensor(X[200:], ms.float32)
test_y = y[200:]
print('Training cases:', 200, 'Test cases:', len(test_y))

## 3. Build the MindSpore model

In [ ]:
class PatientNet(nn.Cell):
    def __init__(self):
        super().__init__()
        self.net = nn.SequentialCell(nn.Dense(10,32), nn.ReLU(), nn.Dense(32,16), nn.ReLU(), nn.Dense(16,2))
    def construct(self, x):
        return self.net(x)

net = PatientNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = nn.Adam(net.trainable_params(), learning_rate=0.001)

def forward_fn(features, labels):
    logits = net(features)
    return loss_fn(logits, labels), logits

grad_fn = ms.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

def train_step(features, labels):
    (loss, logits), grads = grad_fn(features, labels)
    optimizer(grads)
    return loss

print(net)

## 4. Train and evaluate

In [ ]:
for epoch in range(8):
    losses=[]
    for features, labels in train_ds.create_tuple_iterator():
        losses.append(float(train_step(features, labels).asnumpy()))
    if epoch in [0,3,7]: print(f'Epoch {epoch+1}: loss={np.mean(losses):.4f}')

net.set_train(False)
pred = ops.argmax(net(test_x), axis=1).asnumpy()
accuracy = float((pred == test_y).mean())
print('Synthetic test accuracy:', round(accuracy,3))
ms.save_checkpoint(net, 'mindspore_fundamentals.ckpt')

## Reflection
1. What is a tensor? 2. What does the loss measure? 3. Why keep a test set unseen? 4. Why does this synthetic result have no clinical meaning?